# Dual Medical Assistant - Qwen or Mistral

This notebook uses the same Qwen-style RAG assets as `05_qwen_app.ipynb`,
loads the fine-tuned Qwen and Mistral adapters, and adds a model selector to the
existing medical chat interface.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q faiss-cpu sentence-transformers fastapi uvicorn nest-asyncio pyngrok transformers

In [ ]:
from __future__ import annotations

import ast
import gc
import json
import os
import re
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import faiss
import numpy as np
import torch
from google.colab import drive

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/medical_qwen_project")
LEGACY_MISTRAL_PROJECT = Path("/content/drive/MyDrive/medical_chatbot")
LOCAL_PROJECT = Path.cwd()

QWEN_ADAPTER_DIR = DRIVE_PROJECT / "final_model"
MISTRAL_ADAPTER_CANDIDATES = [
    DRIVE_PROJECT / "Mistral_chatbot" / "final_model",
    LEGACY_MISTRAL_PROJECT / "final_model",
    LOCAL_PROJECT / "Mistral_chatbot" / "final_model",
]
RAG_DIR_CANDIDATES = [DRIVE_PROJECT / "rag_assets", LOCAL_PROJECT / "rag_assets"]
UI_HTML_CANDIDATES = [DRIVE_PROJECT / "ui.html", LOCAL_PROJECT / "ui.html"]
MEDICAL_WHITELIST_CANDIDATES = [
    DRIVE_PROJECT / "MEDICAL_WHITELIST.txt",
    LOCAL_PROJECT / "MEDICAL_WHITELIST.txt",
]


def first_existing_path(candidates: Iterable[Path], label: str, must_exist: bool = True) -> Path:
    candidates = [Path(path) for path in candidates]
    for path in candidates:
        if path.exists():
            return path
    if must_exist:
        checked = "\n".join(f"- {path}" for path in candidates)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")
    return candidates[0]


def require_adapter_dir(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} adapter directory not found: {path}")
    if not (path / "adapter_config.json").exists():
        raise FileNotFoundError(f"{label} adapter_config.json not found in: {path}")


MISTRAL_ADAPTER_DIR = first_existing_path(MISTRAL_ADAPTER_CANDIDATES, "Mistral adapter", must_exist=False)
RAG_DIR = first_existing_path(RAG_DIR_CANDIDATES, "RAG assets")
UI_HTML_PATH = first_existing_path(UI_HTML_CANDIDATES, "UI HTML", must_exist=False)

FAISS_INDEX_PATH = RAG_DIR / "medical_rag.faiss"
DOCS_PATH = RAG_DIR / "rag_documents.jsonl"
RETRIEVAL_CONFIG_PATH = RAG_DIR / "retrieval_config.json"
PROMPT_TEMPLATE_PATH = RAG_DIR / "medical_prompt_templates.json"

for required_path, label in [
    (QWEN_ADAPTER_DIR, "Qwen adapter"),
    (FAISS_INDEX_PATH, "FAISS index"),
    (DOCS_PATH, "RAG documents"),
]:
    if not required_path.exists():
        raise FileNotFoundError(f"{label} not found: {required_path}")

print("Qwen adapter:", QWEN_ADAPTER_DIR)
print("Mistral adapter:", MISTRAL_ADAPTER_DIR)
print("RAG assets:", RAG_DIR)
print("UI HTML:", UI_HTML_PATH)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
DTYPE = None
KEEP_MODELS_IN_MEMORY = False
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

MODEL_LABELS = {
    "qwen": "Qwen2.5-7B Medical QLoRA",
    "mistral": "Mistral-7B Medical QLoRA",
}


@dataclass
class LoadedModel:
    key: str
    label: str
    model: Any
    tokenizer: Any


def normalize_model_key(model_key: Optional[str]) -> str:
    key = (model_key or "qwen").strip().lower()
    return key if key in MODEL_LABELS else "qwen"


def adapter_base_model(adapter_dir: Path, fallback: str) -> str:
    config_path = adapter_dir / "adapter_config.json"
    if config_path.exists():
        with config_path.open("r", encoding="utf-8") as handle:
            return json.load(handle).get("base_model_name_or_path") or fallback
    return fallback


def set_tokenizer_padding(tokenizer: Any) -> Any:
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_qwen_model() -> LoadedModel:
    require_adapter_dir(QWEN_ADAPTER_DIR, "Qwen")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(QWEN_ADAPTER_DIR),
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    tokenizer = set_tokenizer_padding(tokenizer)
    model.config.pad_token_id = tokenizer.pad_token_id
    FastLanguageModel.for_inference(model)
    model.eval()
    return LoadedModel("qwen", MODEL_LABELS["qwen"], model, tokenizer)


def load_mistral_model() -> LoadedModel:
    require_adapter_dir(MISTRAL_ADAPTER_DIR, "Mistral")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model_name = adapter_base_model(MISTRAL_ADAPTER_DIR, "mistralai/Mistral-7B-Instruct-v0.2")
    auth_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    tokenizer = AutoTokenizer.from_pretrained(str(MISTRAL_ADAPTER_DIR), use_fast=True, **auth_kwargs)
    tokenizer = set_tokenizer_padding(tokenizer)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        dtype=torch.float16,
        quantization_config=bnb_config,
        **auth_kwargs,
    )
    model = PeftModel.from_pretrained(base_model, str(MISTRAL_ADAPTER_DIR), **auth_kwargs)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return LoadedModel("mistral", MODEL_LABELS["mistral"], model, tokenizer)


MODEL_LOADERS = {"qwen": load_qwen_model, "mistral": load_mistral_model}
_MODEL_CACHE: Dict[str, LoadedModel] = {}
_MODEL_LOCK = threading.RLock()


def unload_models_except(keep_key: Optional[str]) -> None:
    for key in list(_MODEL_CACHE.keys()):
        if key != keep_key:
            bundle = _MODEL_CACHE.pop(key)
            del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_model(model_key: Optional[str]) -> LoadedModel:
    key = normalize_model_key(model_key)
    with _MODEL_LOCK:
        if key in _MODEL_CACHE:
            return _MODEL_CACHE[key]
        if not KEEP_MODELS_IN_MEMORY:
            unload_models_except(None)
        print(f"Loading {MODEL_LABELS[key]}...")
        _MODEL_CACHE[key] = MODEL_LOADERS[key]()
        print(f"Loaded {MODEL_LABELS[key]}.")
        return _MODEL_CACHE[key]


print("Model loaders ready. Run the preload cell before launching the app server.")

In [ ]:
from sentence_transformers import SentenceTransformer

DEFAULT_RETRIEVAL_CONFIG = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "top_k": 4,
    "min_score": 0.20,
    "max_context_chars": 1400,
    "memory_turns": 4,
}


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " ")).strip()


def metadata_dict(metadata: Any) -> Dict[str, Any]:
    if isinstance(metadata, dict):
        return metadata
    if isinstance(metadata, str):
        try:
            parsed = json.loads(metadata)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            return {}
    return {}


def load_retrieval_config() -> Dict[str, Any]:
    if RETRIEVAL_CONFIG_PATH.exists():
        with RETRIEVAL_CONFIG_PATH.open("r", encoding="utf-8") as handle:
            return {**DEFAULT_RETRIEVAL_CONFIG, **json.load(handle)}
    return dict(DEFAULT_RETRIEVAL_CONFIG)


def load_rag_documents(path: Path) -> List[Dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            record["text"] = clean_text(record.get("text", ""))
            record["title"] = clean_text(record.get("title", ""))
            record["source"] = clean_text(record.get("source", ""))
            record["metadata"] = metadata_dict(record.get("metadata", {}))
            records.append(record)
    return records


RETRIEVAL_CONFIG = load_retrieval_config()
EMBEDDING_MODEL_NAME = RETRIEVAL_CONFIG["embedding_model"]
documents = load_rag_documents(DOCS_PATH)
index = faiss.read_index(str(FAISS_INDEX_PATH))
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
if index.ntotal != len(documents):
    print(f"Warning: FAISS vectors ({index.ntotal}) and documents ({len(documents)}) differ.")


def normalize_lookup_text(text: str) -> str:
    text = re.sub(r"[^a-z0-9]+", " ", clean_text(text).lower())
    return f" {text.strip()} "


def query_mentions_drug(query: str, drug_name: str) -> bool:
    drug_name = clean_text(drug_name)
    return len(drug_name) >= 3 and normalize_lookup_text(drug_name) in normalize_lookup_text(query)


def query_has_exact_drug_mention(query: str) -> bool:
    return any(
        query_mentions_drug(query, metadata_dict(record.get("metadata", {})).get("drug_name", ""))
        for record in documents
    )


def infer_requested_doc_types(query: str) -> List[str]:
    text = clean_text(query).lower()
    requested = []
    medication_cause_question = (
        re.search(r"\b(can|could|does|did|will)\b.*\bcause\b", text)
        and (
            re.search(r"\b(it|this|that|medicine|medication|drug|pill|tablet)\b", text)
            or query_has_exact_drug_mention(query)
        )
    )
    if re.search(r"\b(side effect|side effects|adverse|reaction|reactions)\b", text) or medication_cause_question:
        requested.append("side_effect")
    if re.search(r"\b(used for|use for|treat|treats|indication|indications|prescribed for)\b", text):
        requested.append("indication")
    return requested


def record_to_hit(record: Dict[str, Any], score: float, exact_match: bool = False) -> Dict[str, Any]:
    return {
        "score": max(0.0, min(float(score), 1.25)),
        "id": record.get("id", ""),
        "title": record.get("title", ""),
        "text": record.get("text", ""),
        "source": record.get("source", ""),
        "metadata": metadata_dict(record.get("metadata", {})),
        "exact_match": exact_match,
    }


def exact_medication_hits(query: str, requested_doc_types: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    hits = []
    requested_doc_types = requested_doc_types or []
    for record in documents:
        metadata = metadata_dict(record.get("metadata", {}))
        doc_type = clean_text(metadata.get("doc_type", ""))
        if requested_doc_types and doc_type not in requested_doc_types:
            continue
        if query_mentions_drug(query, metadata.get("drug_name", "")):
            hits.append(record_to_hit(record, score=1.25, exact_match=True))
    return hits


STOPWORDS = {
    "about", "after", "also", "because", "been", "can", "could", "does", "from", "have",
    "having", "into", "make", "many", "more", "most", "much", "should", "that", "their",
    "there", "these", "this", "those", "what", "when", "where", "which", "with", "would",
    "your", "side", "effect", "effects", "medical", "context", "dataset", "local", "fact",
    "facts", "include", "includes",
}


def lexical_tokens(text: str) -> set:
    words = re.findall(r"[a-z][a-z0-9]{2,}", clean_text(text).lower())
    return {word for word in words if word not in STOPWORDS}


def lexical_rerank(query: str, hits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    query_terms = lexical_tokens(query)
    requested_doc_types = infer_requested_doc_types(query)
    query_meds = [
        metadata_dict(record.get("metadata", {})).get("drug_name", "")
        for record in documents
        if query_mentions_drug(query, metadata_dict(record.get("metadata", {})).get("drug_name", ""))
    ][:5]
    reranked = []
    for hit in hits:
        hit = dict(hit)
        metadata = metadata_dict(hit.get("metadata", {}))
        hit_terms = lexical_tokens(" ".join([hit.get("title", ""), hit.get("text", ""), metadata.get("drug_name", "")]))
        lexical_score = len(query_terms & hit_terms) / max(1, len(query_terms)) if query_terms else 0.0
        semantic_score = max(0.0, min(float(hit.get("score", 0.0)), 1.0))
        doc_type_boost = 0.08 if requested_doc_types and metadata.get("doc_type") in requested_doc_types else 0.0
        med_boost = 0.25 if query_meds and any(query_mentions_drug(metadata.get("drug_name", ""), med) for med in query_meds) else 0.0
        hit["score"] = max(0.0, min((0.62 * semantic_score) + (0.28 * lexical_score) + doc_type_boost + med_boost, 1.0))
        if hit.get("exact_match"):
            hit["score"] = max(hit["score"], 1.0)
        reranked.append(hit)
    reranked.sort(key=lambda item: (item.get("exact_match", False), item.get("score", 0.0)), reverse=True)
    return reranked


def dedupe_hits(hits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    unique = []
    for hit in hits:
        key = hit.get("id") or hit.get("text")
        if key in seen:
            continue
        seen.add(key)
        unique.append(hit)
    return unique


def retrieve_top_k(query: str, k: Optional[int] = None, min_score: Optional[float] = None) -> List[Dict[str, Any]]:
    query = clean_text(query)
    if not query:
        return []
    k = int(k or RETRIEVAL_CONFIG.get("top_k", 4))
    min_score = float(RETRIEVAL_CONFIG.get("min_score", 0.20) if min_score is None else min_score)
    requested_doc_types = infer_requested_doc_types(query)
    query_embedding = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    search_k = min(max(k * 4, 12), len(documents), index.ntotal)
    scores, indices = index.search(query_embedding, search_k)
    hits = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or int(idx) >= len(documents) or float(score) < min_score:
            continue
        hits.append(record_to_hit(documents[int(idx)], score=float(score)))
    exact_hits = exact_medication_hits(query, requested_doc_types=requested_doc_types)
    if requested_doc_types:
        hits = [
            hit for hit in hits
            if metadata_dict(hit.get("metadata", {})).get("doc_type") in requested_doc_types
            or hit.get("score", 0.0) >= 0.45
        ]
    return lexical_rerank(query, dedupe_hits(exact_hits + hits))[:k]


def clean_context_text(text: str, max_chars: int = 520) -> str:
    text = clean_text(text)
    text = re.sub(r"(?:\[\s*source\b|\bsource\s*\d+\s*:|\bscore\s*=|\bchunk\s*=|\bchunk_index\b|\bfaiss\b)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(Title|Fact|Source)\s*:\s*", "", text, flags=re.IGNORECASE)
    if len(text) > max_chars:
        text = text[:max_chars].rsplit(" ", 1)[0].rstrip(" ,;:") + "..."
    return text.strip(" ,;:")


def build_retrieval_context(hits: List[Dict[str, Any]]) -> str:
    max_chars = int(RETRIEVAL_CONFIG.get("max_context_chars", 1400))
    parts = []
    used = 0
    for hit in hits:
        metadata = metadata_dict(hit.get("metadata", {}))
        label = clean_text(metadata.get("drug_name") or hit.get("title") or "Medical fact")
        part = f"{label}: {clean_context_text(hit.get('text', ''))}"
        if used + len(part) > max_chars:
            remaining = max_chars - used
            if remaining < 120:
                break
            part = part[:remaining].rsplit(" ", 1)[0].rstrip(" ,;:") + "..."
        parts.append(f"- {part}")
        used += len(part)
    return "\n".join(parts) if parts else "No relevant local RAG facts were retrieved."


print(f"Loaded {len(documents):,} RAG documents and {index.ntotal:,} FAISS vectors.")

In [ ]:
DEFAULT_MEDICAL_WHITELIST = [
    "pain", "fever", "cough", "nausea", "vomit", "dizzy", "dizziness", "rash",
    "bleeding", "headache", "migraine", "stomach", "abdomen", "chest", "breathing",
    "shortness of breath", "symptom", "doctor", "hospital", "clinic", "emergency",
    "medicine", "medication", "drug", "pill", "tablet", "dose", "dosage", "side effect",
    "adverse", "infection", "diabetes", "asthma", "allergy", "pregnant", "pregnancy",
    "blood pressure", "heart", "stroke", "depression", "anxiety", "sleep", "fatigue",
    "weakness", "diarrhea", "constipation", "ibuprofen", "aspirin", "amoxicillin",
    "metformin", "paracetamol", "acetaminophen", "antibiotic", "pharmacist",
]


def load_medical_whitelist() -> List[str]:
    for path in MEDICAL_WHITELIST_CANDIDATES:
        if not path.exists():
            continue
        try:
            module = ast.parse(path.read_text(encoding="utf-8"))
            for node in module.body:
                if isinstance(node, ast.Assign):
                    for target in node.targets:
                        if isinstance(target, ast.Name) and target.id == "MEDICAL_WHITELIST":
                            values = ast.literal_eval(node.value)
                            return [clean_text(item).lower() for item in values if clean_text(item)]
        except Exception as exc:
            print(f"Could not parse whitelist at {path}: {exc}")
    return DEFAULT_MEDICAL_WHITELIST


MEDICAL_WHITELIST = list(dict.fromkeys(load_medical_whitelist()))
REFUSAL_MSG = "I'm a specialized medical assistant and can only answer health-related questions. Please ask me about symptoms, medications, or health concerns."
GREETINGS = {"hello", "hi", "hey", "salut", "bonjour", "good morning", "good evening", "hola"}

# Single body-part or anatomy words are weak evidence. They can be medical by
# themselves, but they should not let an unrelated topic bypass the scope gate.
WEAK_MEDICAL_SCOPE_TERMS = {
    "abdomen", "ankle", "anus", "arm", "artery", "back", "bladder", "blood",
    "blood vessel", "bone", "bowel", "brain", "breast", "calf", "cartilage",
    "cervix", "chest", "chin", "colon", "disc", "ear", "eardrum", "elbow",
    "eye", "eyelid", "face", "finger", "foot", "forearm", "forehead", "gas",
    "gums", "hair", "hand", "head", "heart", "heel", "hip", "intestine",
    "jaw", "joint", "kidney", "knee", "leg", "ligament", "lip", "liver",
    "lung", "marrow", "mouth", "muscle", "nail", "neck", "nerve", "nipple",
    "nose", "nostril", "ovary", "pancreas", "pelvis", "penis", "pupil",
    "rectum", "retina", "rib", "scalp", "shin", "shoulder", "sinus", "skin",
    "spine", "spleen", "stomach", "temple", "tendon", "testicle", "thigh",
    "throat", "thumb", "thyroid", "toe", "tongue", "tooth", "teeth", "uterus",
    "vagina", "vein", "vertebra", "wrist",
}

NON_MEDICAL_TOPIC_PATTERNS = {
    "programming": r"\b(react(?:\.js|js)|react\s+(?:component|app|hooks?|state|router|framework|library|developer)|javascript|typescript|python|java|html|css|node(?:\.js)?|next(?:\.js)?|vue|angular|django|flask|fastapi|api|database|sql|frontend|backend|component|function|class|bug|debug|coding|programming|software|website|web\s+app|git|github)\b",
    "math": r"\b(algebra|calculus|geometry|equation|integral|derivative|matrix|probability|statistics)\b",
    "finance": r"\b(stocks?|crypto|bitcoin|forex|investment|portfolio|banking|loan|mortgage|taxes?)\b",
    "sports": r"\b(football|soccer|basketball|tennis|cricket|baseball|match|league|tournament|score)\b",
    "entertainment": r"\b(movie|movies|song|songs|lyrics|album|anime|game|gaming|netflix|celebrity)\b",
}

FOCUSED_NON_MEDICAL_PATTERNS = [
    r"\b(?:what|who|where|when|why|how)\s+(?:is|are|do|does|can)\s+(?:react(?:\.js|js)|react\s+(?:component|app|hooks?|state|router|framework|library|developer)|javascript|typescript|python|java|html|css|node(?:\.js)?|next(?:\.js)?|vue|angular|api|programming|coding|software|framework|library)\b",
    r"\b(?:explain|define|teach|write|build|create|code|debug)\b.*\b(?:react(?:\.js|js)|react\s+(?:component|app|hooks?|state|router|framework|library|developer)|javascript|typescript|python|java|html|css|node(?:\.js)?|next(?:\.js)?|api|website|web\s+app|component|function|class)\b",
]

MEDICAL_INTENT_PATTERNS = [
    r"\b(symptoms?|diagnos(?:e|is|ed|ing)?|treat(?:ment|ed|ing|s)?|prescri(?:be|bed|ption)|medicine|medication|drug|dose|dosage|side\s*effects?|adverse|contraindication|interaction|allerg(?:y|ic)|infection|disease|condition|syndrome|doctor|hospital|clinic|emergency|urgent\s+care|pharmacist|pregnan(?:t|cy)|vaccine|surgery|therapy)\b",
    r"\b(pain|ache|aching|hurt(?:s|ing)?|sore|fever|cough|nausea|vomit(?:ing)?|dizz(?:y|iness)|rash|bleed(?:ing)?|swoll(?:en|ing)|itch(?:y|ing)?|burn(?:ing)?|numb(?:ness)?|tingl(?:e|ing)|shortness\s+of\s+breath|trouble\s+breathing|headache|migraine|diarrhea|constipation|fatigue|weakness|palpitations|wheezing|congestion|runny\s+nose|stuffy\s+nose|nosebleed|eye\s+strain)\b",
]

HEALTH_RELATION_PATTERN = r"\b(health|medical|symptoms?|condition|disease|diagnos(?:e|is|ed|ing)?|treat(?:ment|ed|ing|s)?|cause|causes|caused|trigger|triggers|worsen|worsens|risk|safe|safety|side\s*effects?|pain|strain|injury|allerg(?:y|ic)|infection|dose|medicine|medication|drug|doctor|hospital)\b"
NON_MEDICAL_HEALTH_BRIDGE_PATTERN = r"\b(health|medical|symptoms?|diagnos(?:e|is|ed|ing)?|treat(?:ment|ed|ing|s)?|cause|causes|caused|trigger|triggers|worsen|worsens|affect|affects|damage|damages|hurt|hurts|risk|safe|safety|pain|strain|injury|headache|migraine|fatigue|nausea|dizziness|allerg(?:y|ic)|infection|doctor|hospital)\b"


def whitelist_phrase_pattern(phrase: str) -> Optional[re.Pattern]:
    words = [re.escape(part) for part in re.split(r"\s+", clean_text(phrase).lower()) if part]
    if not words:
        return None
    return re.compile(r"(?<![a-z0-9])" + r"\s+".join(words) + r"(?![a-z0-9])", re.IGNORECASE)


MEDICAL_WHITELIST_PATTERNS = [
    (term, pattern)
    for term in MEDICAL_WHITELIST
    if len(term) >= 3
    for pattern in [whitelist_phrase_pattern(term)]
    if pattern is not None
]

EMERGENCY_PATTERNS = {
    "chest pain": [r"\bchest\s+(pain|pressure|tightness|crushing|heaviness)\b", r"\bpain\s+in\s+my\s+chest\b"],
    "shortness of breath": [r"\bshort(ness)?\s+of\s+breath\b", r"\b(can'?t|cannot)\s+breathe\b", r"\btrouble\s+breathing\b"],
    "stroke symptoms": [r"\bstroke\b", r"\bslurred\s+speech\b", r"\bone[-\s]?sided\s+(weakness|numbness)\b"],
    "overdose or poisoning": [r"\boverdose\b", r"\btook\s+too\s+much\b", r"\btoo\s+many\s+pills\b", r"\bpoison(ed|ing)?\b"],
    "severe allergic reaction": [r"\banaphylaxis\b", r"\bthroat\s+swelling\b", r"\b(tongue|lip|face)\s+swelling\b"],
    "suicidal intent": [r"\bkill\s+myself\b", r"\bend\s+my\s+life\b", r"\bsuicid(e|al)\b", r"\bwant\s+to\s+die\b", r"\bself[-\s]?harm\b"],
    "major bleeding": [r"\bheavy\s+bleeding\b", r"\bbleeding\s+(won'?t|will\s+not)\s+stop\b", r"\bvomiting\s+blood\b"],
    "loss of consciousness": [r"\blost\s+consciousness\b", r"\bpassed\s+out\b", r"\bfainted\b", r"\bunresponsive\b"],
}

UNSAFE_REQUEST_PATTERNS = {
    "dangerous_dosage": r"\b(lethal dose|fatal dose|how many pills|how much .* to overdose|unsafe dose|maximum to get high)\b",
    "drug_misuse": r"\b(get high|abuse|recreational|snort|inject|crush.*pill|opioid high|fentanyl high)\b",
    "controlled_prescribing": r"\b(write me a prescription|fake prescription|controlled substance|opioid|oxycodone|hydrocodone|xanax)\b",
    "rule_override": r"\b(ignore (all )?(previous|safety) instructions|pretend you are a doctor|act as my doctor|no disclaimers)\b",
}


def emergency_categories(query: str) -> List[str]:
    text = clean_text(query).lower()
    return [category for category, patterns in EMERGENCY_PATTERNS.items() if any(re.search(pattern, text) for pattern in patterns)]


def emergency_response(categories: List[str]) -> str:
    if "suicidal intent" in categories:
        return "I'm really sorry you're dealing with this. If you might hurt yourself or cannot stay safe, call emergency services now, or contact a crisis line such as 988 in the US or your local emergency number."
    if "overdose or poisoning" in categories:
        return "This may be an overdose or poisoning emergency. Call emergency services or poison control now. Do not wait for symptoms to get worse."
    if "chest pain" in categories or "shortness of breath" in categories:
        return "Chest pain, chest pressure, or serious breathing trouble can be an emergency. Call your local emergency number now, especially with sweating, fainting, blue lips, confusion, pain spreading to the arm or jaw, or trouble speaking full sentences."
    return "What you described could be urgent. Please call local emergency services now if symptoms are severe, sudden, worsening, or involve fainting, confusion, major bleeding, severe allergic reaction, weakness, or trouble breathing."


def unsafe_request_check(query: str) -> Dict[str, Any]:
    text = clean_text(query).lower()
    matches = [name for name, pattern in UNSAFE_REQUEST_PATTERNS.items() if re.search(pattern, text)]
    return {"is_unsafe": bool(matches), "matches": matches}


def unsafe_request_response(check: Dict[str, Any]) -> str:
    if "rule_override" in check.get("matches", []):
        return "I cannot ignore medical safety rules or pretend to be a licensed clinician. I can still help with cautious medical education and safer next steps."
    return "I cannot help with dangerous dosing, drug misuse, or instructions that could cause harm. If this involves a possible overdose or poisoning, contact emergency services or poison control now."


def exact_whitelist_matches(query: str) -> List[str]:
    text = clean_text(query).lower()
    return [term for term, pattern in MEDICAL_WHITELIST_PATTERNS if pattern.search(text)]


def non_medical_topic_matches(query: str) -> List[str]:
    text = clean_text(query).lower()
    return [name for name, pattern in NON_MEDICAL_TOPIC_PATTERNS.items() if re.search(pattern, text)]


def has_focused_non_medical_topic(query: str) -> bool:
    text = clean_text(query).lower()
    return any(re.search(pattern, text) for pattern in FOCUSED_NON_MEDICAL_PATTERNS)


def has_medical_intent(query: str) -> bool:
    text = clean_text(query).lower()
    return any(re.search(pattern, text) for pattern in MEDICAL_INTENT_PATTERNS)


def has_health_relationship(query: str) -> bool:
    return bool(re.search(HEALTH_RELATION_PATTERN, clean_text(query).lower()))


def has_non_medical_health_bridge(query: str) -> bool:
    return bool(re.search(NON_MEDICAL_HEALTH_BRIDGE_PATTERN, clean_text(query).lower()))


def rag_supports_medical_query(hits: Optional[List[Dict[str, Any]]]) -> bool:
    if not hits:
        return False
    top_hit = hits[0]
    score = float(top_hit.get("score", 0.0))
    return bool(top_hit.get("exact_match")) or score >= 0.55


def medical_scope_check(query: str, hits: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    whitelist_matches = exact_whitelist_matches(query)
    strong_matches = [term for term in whitelist_matches if term not in WEAK_MEDICAL_SCOPE_TERMS]
    non_medical_matches = non_medical_topic_matches(query)
    focused_non_medical = has_focused_non_medical_topic(query)
    medical_intent = has_medical_intent(query) or bool(strong_matches)
    health_relationship = has_health_relationship(query) and bool(whitelist_matches or medical_intent)
    health_bridge = has_non_medical_health_bridge(query) and bool(whitelist_matches or medical_intent)
    rag_support = rag_supports_medical_query(hits)

    if focused_non_medical:
        return {
            "is_medical": False,
            "reason": "focused_non_medical_topic",
            "whitelist_matches": whitelist_matches,
            "non_medical_matches": non_medical_matches,
        }

    if non_medical_matches and not (medical_intent and health_bridge):
        return {
            "is_medical": False,
            "reason": "mixed_non_medical_topic",
            "whitelist_matches": whitelist_matches,
            "non_medical_matches": non_medical_matches,
        }

    is_medical = bool(medical_intent or whitelist_matches or rag_support)
    return {
        "is_medical": is_medical,
        "reason": "accepted" if is_medical else "no_medical_signal",
        "whitelist_matches": whitelist_matches,
        "non_medical_matches": non_medical_matches,
    }


def is_medical_query(query: str, hits: Optional[List[Dict[str, Any]]] = None) -> bool:
    return medical_scope_check(query, hits).get("is_medical", False)


DEFAULT_MEDICAL_SYSTEM_PROMPT = '''
You are a calm, conversational medical education assistant. You are not a licensed doctor and you do not pretend to be one.

Core behavior:
- Ask useful follow-up questions for vague or mild symptoms before narrowing possibilities.
- Explain uncertainty naturally.
- Use retrieved facts internally for grounding. Do not expose raw retrieval blocks, scores, source labels, or database-style lists.
- Refuse non-medical questions even when they contain an isolated medical or anatomy word.
- For medications, do not invent dose, interactions, contraindications, pregnancy safety, kidney/liver safety, or personalized treatment decisions.
- Escalate clearly only for real red flags.

Style:
- Prefer 1 to 3 short paragraphs, or a few bullets when asking clarifying questions.
- Use plain language.
'''.strip()

DEFAULT_USER_PROMPT_TEMPLATE = '''
Recent conversation:
{conversation_context}

User question:
{user_query}

Medical context for internal grounding only:
{retrieval_context}

Behavior directive:
{behavior_directive}

Scope and safety note:
{scope_note}

Write the answer to the user. Do not mention retrieval, scores, source labels, chunks, or raw context blocks.
'''.strip()

MEDICAL_DISCLAIMER = "This chatbot is for educational support only and is not a substitute for professional medical care. For emergencies, call local emergency services."


def load_prompt_templates() -> Dict[str, str]:
    if PROMPT_TEMPLATE_PATH.exists():
        with PROMPT_TEMPLATE_PATH.open("r", encoding="utf-8") as handle:
            loaded = json.load(handle)
        return {
            "medical_system_prompt": loaded.get("medical_system_prompt", DEFAULT_MEDICAL_SYSTEM_PROMPT),
            "user_prompt_template": loaded.get("user_prompt_template", DEFAULT_USER_PROMPT_TEMPLATE),
            "medical_disclaimer": loaded.get("medical_disclaimer", MEDICAL_DISCLAIMER),
        }
    return {"medical_system_prompt": DEFAULT_MEDICAL_SYSTEM_PROMPT, "user_prompt_template": DEFAULT_USER_PROMPT_TEMPLATE, "medical_disclaimer": MEDICAL_DISCLAIMER}


PROMPTS = load_prompt_templates()


def conversation_context(history: Optional[List[Any]]) -> str:
    turns = []
    for turn in history or []:
        if isinstance(turn, (list, tuple)) and len(turn) >= 2:
            user_text, assistant_text = clean_text(turn[0]), clean_text(turn[1])
            if user_text and assistant_text:
                turns.append((user_text, assistant_text))
    turns = turns[-int(RETRIEVAL_CONFIG.get("memory_turns", 4)):]
    if not turns:
        return "None"
    return "\n".join(f"User: {u}\nAssistant: {a}" for u, a in turns)


def build_messages(user_query: str, history: Optional[List[Any]], retrieval_context: str, hits: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    user_prompt = PROMPTS["user_prompt_template"].format(
        conversation_context=conversation_context(history),
        user_query=clean_text(user_query),
        retrieval_context=retrieval_context,
        behavior_directive="Use the retrieved medical facts only where relevant, and explain uncertainty clearly." if hits else "Ask concise follow-up questions when the question is vague.",
        scope_note=PROMPTS["medical_disclaimer"],
    )
    return [{"role": "system", "content": PROMPTS["medical_system_prompt"]}, {"role": "user", "content": user_prompt}]


def model_input_device(model: Any) -> torch.device:
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def stream_answer_tokens(model_key: Optional[str], user_message: str, history: Optional[List[Any]] = None):
    user_message = clean_text(user_message)
    if not user_message:
        yield "Please enter a medical question."
        return
    if user_message.lower() in GREETINGS:
        yield "Hello. I am your specialized medical AI assistant. How can I help with your health question today?"
        return
    categories = emergency_categories(user_message)
    if categories:
        yield emergency_response(categories)
        return
    unsafe_check = unsafe_request_check(user_message)
    if unsafe_check["is_unsafe"]:
        yield unsafe_request_response(unsafe_check)
        return

    hits = retrieve_top_k(user_message)
    if not is_medical_query(user_message, hits):
        yield REFUSAL_MSG
        return

    bundle = get_model(model_key)
    tokenizer, model = bundle.tokenizer, bundle.model
    messages = build_messages(user_message, history, build_retrieval_context(hits), hits)

    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
        if input_ids.shape[-1] > MAX_SEQ_LENGTH:
            input_ids = input_ids[:, -MAX_SEQ_LENGTH:]
        generation_inputs = {"input_ids": input_ids.to(model_input_device(model))}
    else:
        prompt_text = "\n\n".join(f"{m['role'].upper()}: {m['content']}" for m in messages) + "\n\nASSISTANT:"
        generation_inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model_input_device(model))

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **generation_inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.05,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    def run_generation() -> None:
        with torch.no_grad():
            model.generate(**generation_kwargs)

    thread = threading.Thread(target=run_generation)
    thread.start()
    for token in streamer:
        yield token
    thread.join()


print(f"Safety and prompt layer ready. Medical whitelist terms: {len(MEDICAL_WHITELIST):,}")

In [ ]:
# Preload the startup model before launching the blocking app server cell.
# To preload Mistral instead, set PRELOAD_MODELS = "mistral".
# Loading both 7B adapters at once requires enough GPU memory and KEEP_MODELS_IN_MEMORY = True.
PRELOAD_MODELS = str(globals().get("PRELOAD_MODELS", os.environ.get("PRELOAD_MODELS", "qwen"))).strip()
PRELOAD_MODEL_KEYS = [
    normalize_model_key(part)
    for part in re.split(r"[,\s]+", PRELOAD_MODELS)
    if part.strip()
]
if not PRELOAD_MODEL_KEYS:
    PRELOAD_MODEL_KEYS = ["qwen"]

PRELOAD_MODEL_KEYS = list(dict.fromkeys(PRELOAD_MODEL_KEYS))
if len(PRELOAD_MODEL_KEYS) > 1 and not KEEP_MODELS_IN_MEMORY:
    print("KEEP_MODELS_IN_MEMORY is False; preloading only the first requested model to avoid GPU memory issues.")
    PRELOAD_MODEL_KEYS = PRELOAD_MODEL_KEYS[:1]

for model_key in PRELOAD_MODEL_KEYS:
    get_model(model_key)

print("Startup model preload complete:", ", ".join(MODEL_LABELS[key] for key in PRELOAD_MODEL_KEYS))

In [ ]:
import asyncio
import nest_asyncio
import uvicorn
from google.colab.output import eval_js
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, StreamingResponse
from pyngrok import ngrok

nest_asyncio.apply()

DUAL_MODEL_CSS = '''
  .model-select-label { display:block; font-size:11px; color:var(--slate-400); margin-bottom:6px; }
  .model-select { width:100%; height:38px; border:1px solid var(--slate-200); border-radius:var(--radius-md); background:var(--slate-0); color:var(--slate-800); font-family:var(--font-body); font-size:13px; padding:0 10px; outline:none; }
  .model-select:focus { border-color:var(--teal-300); box-shadow:0 0 0 3px rgba(45,184,148,.16); }
'''

MODEL_SELECTOR_HTML = '''
    <div class="sidebar-section">
      <h4>Choose model</h4>
      <label class="model-select-label" for="modelSelect">Answering model</label>
      <select id="modelSelect" class="model-select">
        <option value="qwen">Qwen fine-tuned RAG</option>
        <option value="mistral">Mistral fine-tuned RAG</option>
      </select>
    </div>
'''

FALLBACK_UI_HTML = '''
<!DOCTYPE html><html><head><meta charset="UTF-8"><title>Dual Medical Assistant</title>
<style>body{font-family:Arial,sans-serif;margin:0;background:#F7F8FA;color:#1E2533}.wrap{max-width:920px;margin:0 auto;padding:24px}.bar{display:flex;gap:12px;align-items:center;margin-bottom:16px}.chat{height:62vh;overflow:auto;background:#fff;border:1px solid #DDE1E9;border-radius:10px;padding:16px}.msg{margin:12px 0;padding:10px 12px;border-radius:10px}.user{background:#0B8F6E;color:#fff;margin-left:20%}.ai{background:#EEF0F4;margin-right:20%}textarea{width:100%;height:70px;margin-top:12px}button,select{height:38px}</style></head>
<body><div class="wrap"><div class="bar"><strong>Dual Medical Assistant</strong><select id="modelSelect"><option value="qwen">Qwen fine-tuned RAG</option><option value="mistral">Mistral fine-tuned RAG</option></select></div><div id="messages" class="chat"></div><textarea id="input" placeholder="Ask a medical question..."></textarea><button onclick="sendMessage()">Send</button><button onclick="clearChat()">Clear</button></div>
<script>
const API_URL='/chat'; let history=[];
function selectedModel(){return document.getElementById('modelSelect').value}
function add(cls,text){const d=document.createElement('div');d.className='msg '+cls;d.innerHTML=text.replace(/\n/g,'<br>');document.getElementById('messages').appendChild(d);document.getElementById('messages').scrollTop=999999;return d}
function clearChat(){history=[];document.getElementById('messages').innerHTML=''}
async function sendMessage(){const input=document.getElementById('input');const text=input.value.trim();if(!text)return;input.value='';add('user',text);const bubble=add('ai','');const res=await fetch(API_URL,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:text,history:history,model:selectedModel()})});const reader=res.body.getReader();const decoder=new TextDecoder();let full='';while(true){const {done,value}=await reader.read();if(done)break;const lines=decoder.decode(value,{stream:true}).split('\n');for(const line of lines){if(line.startsWith('data: ')){let token=line.slice(6);if(token==='[DONE]')break;token=token.replace(/\\n/g,'\n');full+=token;bubble.innerHTML=full.replace(/\n/g,'<br>')}}}history.push([text,full])}
</script></body></html>
'''


def build_dual_model_ui() -> str:
    html_doc = UI_HTML_PATH.read_text(encoding="utf-8") if UI_HTML_PATH.exists() else FALLBACK_UI_HTML
    html_doc = html_doc.replace("<title>Qwen Medical Assistant</title>", "<title>Dual Medical Assistant</title>")
    html_doc = html_doc.replace("Qwen Medical Assistant", "Dual Medical Assistant")
    html_doc = html_doc.replace("Qwen2.5-7B-Instruct", "Qwen2.5 / Mistral 7B")
    html_doc = html_doc.replace("QLoRA Â· 4-bit quantized", "QLoRA / 4-bit quantized")
    html_doc = html_doc.replace("FAISS Â· top-3 retrieval", "FAISS / shared RAG")
    html_doc = html_doc.replace("Model online", "Backend online")
    html_doc = re.sub(r"\b(?:const|let|var)\s+API_URL\s*=\s*['\"][^'\"]*['\"]\s*;?", "const API_URL = '/chat';", html_doc)
    html_doc = re.sub(r"fetch\(\s*['\"][^'\"]*/chat['\"]", "fetch('/chat'", html_doc)
    html_doc = html_doc.replace("fetch(API_URL,", "fetch('/chat',")
    if ".model-select" not in html_doc:
        html_doc = html_doc.replace("</style>", DUAL_MODEL_CSS + "\n</style>")
    if 'id="modelSelect"' not in html_doc:
        marker = '    <div class="sidebar-section">\n      <h4>Model details</h4>'
        if marker in html_doc:
            html_doc = html_doc.replace(marker, MODEL_SELECTOR_HTML + "\n" + marker, 1)
        else:
            html_doc = html_doc.replace('<aside class="sidebar">', '<aside class="sidebar">\n' + MODEL_SELECTOR_HTML, 1)
    if "function selectedModel()" not in html_doc:
        html_doc = html_doc.replace(
            "let isLoading = false;",
            "let isLoading = false;\n\n  function selectedModel() {\n    const select = document.getElementById('modelSelect');\n    return select ? select.value : 'qwen';\n  }",
            1,
        )
    html_doc = html_doc.replace(
        "body: JSON.stringify({ message: text, history: history })",
        "body: JSON.stringify({ message: text, history: history, model: selectedModel() })",
    )
    html_doc = html_doc.replace("fullText += token;", "fullText += token.replace(/\\\\n/g, '\\n');")
    html_doc = html_doc.replace("3 sources retrieved from FAISS", "FAISS context retrieved")
    return html_doc


app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/", response_class=HTMLResponse)
async def serve_ui():
    return HTMLResponse(build_dual_model_ui())


@app.post("/chat")
async def chat_endpoint(request: Request):
    body = await request.json()
    user_msg = body.get("message", "")
    history = body.get("history", [])
    model_key = normalize_model_key(body.get("model", "qwen"))

    async def token_stream():
        try:
            for token in stream_answer_tokens(model_key, user_msg, history):
                safe = str(token).replace("\r", "").replace("\n", "\\n")
                yield f"data: {safe}\n\n"
                await asyncio.sleep(0)
        except Exception as exc:
            yield f"data: Backend error: {exc}\n\n"
        finally:
            yield "data: [DONE]\n\n"

    return StreamingResponse(token_stream(), media_type="text/event-stream")


SERVER_PORT = 8000


def colab_proxy_url(port: int) -> str:
    try:
        return eval_js(f"google.colab.kernel.proxyPort({port})")
    except Exception:
        return ""


def preload_startup_models() -> List[str]:
    preload_models = str(globals().get("PRELOAD_MODELS", os.environ.get("PRELOAD_MODELS", "qwen"))).strip()
    preload_keys = [
        normalize_model_key(part)
        for part in re.split(r"[,\s]+", preload_models)
        if part.strip()
    ]
    if not preload_keys:
        preload_keys = ["qwen"]
    preload_keys = list(dict.fromkeys(preload_keys))
    if len(preload_keys) > 1 and not KEEP_MODELS_IN_MEMORY:
        print("KEEP_MODELS_IN_MEMORY is False; preloading only the first requested model to avoid GPU memory issues.")
        preload_keys = preload_keys[:1]
    print("Preloading startup model before opening the public URL:", ", ".join(MODEL_LABELS[key] for key in preload_keys))
    for model_key in preload_keys:
        get_model(model_key)
    print("Startup model preload complete:", ", ".join(MODEL_LABELS[key] for key in preload_keys))
    if "mistral" not in preload_keys:
        print("Mistral will load on the first Mistral request. To load it before launch, set PRELOAD_MODELS = 'mistral' before running this cell.")
    return preload_keys


STARTUP_MODEL_KEYS = preload_startup_models()


public_url = ""
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
if NGROK_AUTH_TOKEN:
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        ngrok.kill()
    except Exception:
        pass
    try:
        public_url = str(ngrok.connect(SERVER_PORT))
        print("=" * 55)
        print("Dual medical assistant UI is live via ngrok:")
        print(public_url)
        print("=" * 55)
    except Exception as exc:
        print("ngrok did not start:", exc)
else:
    print("NGROK_AUTH_TOKEN not set. Falling back to Colab proxy.")

if not public_url:
    public_url = colab_proxy_url(SERVER_PORT)
    if public_url:
        print("=" * 55)
        print("Dual medical assistant UI is live via Colab proxy:")
        print(public_url)
        print("=" * 55)
    else:
        print(f"Local server: http://127.0.0.1:{SERVER_PORT}")

loop = asyncio.get_event_loop()
previous_server = globals().get("_DUAL_MEDICAL_SERVER")
previous_task = globals().get("_DUAL_MEDICAL_SERVER_TASK")
if previous_server is not None and previous_task is not None and not previous_task.done():
    print("Stopping previous FastAPI server on this kernel...")
    previous_server.should_exit = True
    loop.run_until_complete(asyncio.sleep(0.5))


def report_server_task_result(task):
    try:
        task.result()
    except asyncio.CancelledError:
        pass
    except KeyboardInterrupt:
        print("FastAPI server stopped.")
    except Exception as exc:
        print("FastAPI server stopped with error:", exc)


config = uvicorn.Config(app=app, host="0.0.0.0", port=SERVER_PORT, loop="asyncio")
server = uvicorn.Server(config=config)
_DUAL_MEDICAL_SERVER = server
_DUAL_MEDICAL_SERVER_TASK = loop.create_task(server.serve())
_DUAL_MEDICAL_SERVER_TASK.add_done_callback(report_server_task_result)
print(f"FastAPI server is running in the background on port {SERVER_PORT}.")